# Context Metrics

This notebook creates sub-national context metrics for the national tool.

It is designed to grow section by section. Each section should create a tidy metrics table keyed by `adm_id`. The final section combines all available context metrics and exports one CSV.

## 0. Setup

Set the country, administrative level, and input/output paths here. Country folders use ISO3 codes.

In [9]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

pd.set_option("display.max_columns", 100)

In [10]:
# Core parameters
COUNTRY_ISO3 = "KEN"
COUNTRY_NAME = "Kenya"
ADMIN_LEVEL = "adm1"  # options currently available: adm0, adm1, adm2
MODULE = "context"
HAZARD = "none"
WORLDPOP_YEAR = 2025

# Resolve paths whether the notebook is run from the repo root or notebooks/.
NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == "notebooks" else NOTEBOOK_CWD

BOUNDARY_PATH = REPO_ROOT / "data" / "boundaries" / COUNTRY_ISO3 / ADMIN_LEVEL / f"{COUNTRY_ISO3}_{ADMIN_LEVEL}.shp"
WORLDPOP_DIR = REPO_ROOT / "data" / "raw" / COUNTRY_ISO3 / MODULE / "worldpop"
OUTPUT_DIR = REPO_ROOT / "results" / COUNTRY_ISO3 / MODULE
OUTPUT_PATH = OUTPUT_DIR / f"{COUNTRY_ISO3}_{ADMIN_LEVEL}_{MODULE}_metrics.csv"

BOUNDARY_PATH, WORLDPOP_DIR, OUTPUT_PATH

(WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/boundaries/KEN/adm1/KEN_adm1.shp'),
 WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/data/raw/KEN/context/worldpop'),
 WindowsPath('C:/Users/Mark.DESKTOP-UFHIN6T/Projects/national_tool_metrics/results/KEN/context/KEN_adm1_context_metrics.csv'))

## 0.1 Load Administrative Boundaries

The current Kenya boundary files use `shapeID` as the stable administrative identifier and `shapeName` as the display name.

In [11]:
ADMIN_ID_FIELD = "shapeID"
ADMIN_NAME_FIELD = "shapeName"

boundaries = gpd.read_file(BOUNDARY_PATH)

required_boundary_cols = {ADMIN_ID_FIELD, ADMIN_NAME_FIELD, "geometry"}
missing_boundary_cols = required_boundary_cols.difference(boundaries.columns)
if missing_boundary_cols:
    raise ValueError(f"Boundary layer is missing required columns: {sorted(missing_boundary_cols)}")

admin_regions = boundaries[[ADMIN_ID_FIELD, ADMIN_NAME_FIELD, "geometry"]].copy()
admin_regions = admin_regions.rename(columns={ADMIN_ID_FIELD: "adm_id", ADMIN_NAME_FIELD: "adm_name"})
admin_regions["country_iso3"] = COUNTRY_ISO3
admin_regions["country_name"] = COUNTRY_NAME
admin_regions["admin_level"] = ADMIN_LEVEL.upper()
admin_regions["module"] = MODULE
admin_regions["hazard"] = HAZARD

admin_regions.head()

,adm_id,adm_name,geometry,country_iso3,country_name,admin_level,module,hazard
0,32016919B72266624462344,Turkana,"POLYGON ((35.94724 4.63093, 35.94857 4.62942, ...",KEN,Kenya,ADM1,context,none
1,32016919B63496705134089,Marsabit,"POLYGON ((36.71131 4.44592, 36.72669 4.44664, ...",KEN,Kenya,ADM1,context,none
2,32016919B2031803566233,Mandera,"POLYGON ((39.77523 3.6681, 39.86773 3.8675, 39...",KEN,Kenya,ADM1,context,none
3,32016919B89873713911655,Wajir,"POLYGON ((39.31809 3.47197, 39.33258 3.46684, ...",KEN,Kenya,ADM1,context,none
4,32016919B96045830258165,West Pokot,"POLYGON ((34.93152 2.43647, 34.93121 2.43842, ...",KEN,Kenya,ADM1,context,none


## 0.2 Helper Functions

The population rasters are summarized using zonal sums. By default, `all_touched=False`, so raster cells are included when their centre falls inside each administrative region. This is a simple and reproducible first pass.

In [12]:
def check_path_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def percentage(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    """Return numerator / denominator * 100, with missing values where denominator is zero."""
    denominator = denominator.replace({0: np.nan})
    return numerator / denominator * 100


def zonal_raster_sum(raster_path: Path, polygons: gpd.GeoDataFrame, all_touched: bool = False) -> pd.Series:
    """Sum raster values within each polygon and return a Series aligned to polygons."""
    check_path_exists(raster_path, "Raster")
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        nodata = src.nodata

    polygons_for_stats = polygons
    if raster_crs is not None and polygons.crs != raster_crs:
        polygons_for_stats = polygons.to_crs(raster_crs)

    stats = zonal_stats(
        polygons_for_stats,
        str(raster_path),
        stats=["sum"],
        nodata=nodata,
        all_touched=all_touched,
    )
    return pd.Series([item.get("sum", 0) or 0 for item in stats], index=polygons.index, dtype="float64")


def parse_worldpop_age_sex_rasters(worldpop_dir: Path, country_iso3: str, year: int) -> pd.DataFrame:
    """Create an inventory of WorldPop age-sex raster files."""
    country_lower = country_iso3.lower()
    pattern = re.compile(rf"^{country_lower}_(?P<sex>[fmt])_(?P<age>\d{{2}})_{year}_.*\.tif$", re.IGNORECASE)
    records = []

    for path in sorted(worldpop_dir.glob("*.tif")):
        match = pattern.match(path.name)
        if match:
            records.append(
                {
                    "sex": match.group("sex").lower(),
                    "age_start": int(match.group("age")),
                    "age_code": match.group("age"),
                    "path": path,
                }
            )

    inventory = pd.DataFrame.from_records(records)
    if inventory.empty:
        raise FileNotFoundError(f"No WorldPop age-sex rasters found in {worldpop_dir}")
    return inventory.sort_values(["sex", "age_start"]).reset_index(drop=True)


def find_total_population_raster(worldpop_dir: Path, country_iso3: str, year: int) -> Path:
    country_lower = country_iso3.lower()
    matches = sorted(worldpop_dir.glob(f"{country_lower}_pop_{year}_*.tif"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one total population raster, found {len(matches)} in {worldpop_dir}")
    return matches[0]


def sum_age_group(
    inventory: pd.DataFrame,
    sex: str,
    age_starts: list[int],
    polygons: gpd.GeoDataFrame,
) -> pd.Series:
    """Sum a set of WorldPop age-band rasters for one sex category."""
    selected = inventory[(inventory["sex"] == sex) & (inventory["age_start"].isin(age_starts))]
    found_ages = set(selected["age_start"])
    missing_ages = sorted(set(age_starts).difference(found_ages))
    if missing_ages:
        raise FileNotFoundError(f"Missing WorldPop rasters for sex={sex}, ages={missing_ages}")

    total = pd.Series(0.0, index=polygons.index)
    for raster_path in selected["path"]:
        total = total + zonal_raster_sum(raster_path, polygons)
    return total

## 1. Population And Demographics

This section summarizes WorldPop population rasters to the selected administrative level.

Metrics being reported:

- Total population.
- Female and male population counts.
- Children under 5.
- School age children (5-14)
- Working-age population aged 15-64.
- Female population aged 15-49.
- Older population aged 65+.

WorldPop age-band naming uses `00` for under 1, `01` for ages 1-4, then 5-year age bands from `05` to `90`.

In [13]:
check_path_exists(BOUNDARY_PATH, "Boundary layer")
check_path_exists(WORLDPOP_DIR, "WorldPop directory")

worldpop_inventory = parse_worldpop_age_sex_rasters(WORLDPOP_DIR, COUNTRY_ISO3, WORLDPOP_YEAR)
total_population_raster = find_total_population_raster(WORLDPOP_DIR, COUNTRY_ISO3, WORLDPOP_YEAR)

worldpop_inventory.groupby("sex")["age_start"].apply(list), total_population_raster.name

(sex
 f    [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, ...
 m    [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, ...
 t    [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, ...
 Name: age_start, dtype: object,
 'ken_pop_2025_CN_1km_R2025A_UA_v1.tif')

In [14]:
ALL_AGE_STARTS = sorted(worldpop_inventory["age_start"].unique().tolist())

AGE_UNDER_5 = [0, 1]
AGE_SCHOOL_CHILDREN_5_14 = [5, 10]
AGE_WORKING_15_64 = list(range(15, 65, 5))
AGE_OLDER_65_PLUS = list(range(65, 95, 5))
AGE_FEMALE_15_49 = list(range(15, 50, 5))

population_metrics = admin_regions[["adm_id"]].copy()

# Total population from the WorldPop total population raster.
population_metrics["pop_total"] = zonal_raster_sum(total_population_raster, admin_regions)

# Sex breakdowns from age-sex rasters.
population_metrics["pop_female"] = sum_age_group(worldpop_inventory, "f", ALL_AGE_STARTS, admin_regions)
population_metrics["pop_male"] = sum_age_group(worldpop_inventory, "m", ALL_AGE_STARTS, admin_regions)

# Age breakdowns from total age-band rasters.
population_metrics["pop_under_5"] = sum_age_group(worldpop_inventory, "t", AGE_UNDER_5, admin_regions)
population_metrics["pop_school_children_5_14"] = sum_age_group(worldpop_inventory, "t", AGE_SCHOOL_CHILDREN_5_14, admin_regions)
population_metrics["pop_working_age_15_64"] = sum_age_group(worldpop_inventory, "t", AGE_WORKING_15_64, admin_regions)
population_metrics["pop_older_65_plus"] = sum_age_group(worldpop_inventory, "t", AGE_OLDER_65_PLUS, admin_regions)

# Additional useful sex-age metrics.
population_metrics["pop_female_childbearing_15_49"] = sum_age_group(worldpop_inventory, "f", AGE_FEMALE_15_49, admin_regions)

population_metrics.head()

,adm_id,pop_total,pop_female,pop_male,pop_under_5,pop_school_children_5_14,pop_working_age_15_64,pop_older_65_plus,pop_female_childbearing_15_49
0,32016919B72266624462344,1.074106e+06,518061.122345,556044.553513,151482.689453,305022.468750,596109.131836,21491.445099,255517.524414
1,32016919B63496705134089,6.150882e+05,287648.498199,327438.485016,91460.078125,180124.093750,329090.162109,14412.233093,132223.533203
2,32016919B2031803566233,1.032164e+06,512839.462677,519317.328094,192278.976562,345974.343750,482180.638672,11722.385681,215100.789062
3,32016919B89873713911655,9.020463e+05,420830.742584,481212.463654,141367.371094,286673.984375,463422.831055,10578.886322,196937.869141
4,32016919B96045830258165,7.387396e+05,371952.902466,366780.400208,134426.886719,222575.203125,364448.130859,17283.093384,163925.096680


## 2. Future Context Sections

Add future context sections below this point. Each section should produce a DataFrame with one row per administrative region and `adm_id` as the join key.

Potential future sections:

- Relative wealth index.
- Infrastructure access or coverage.
- Health, education, or service facility counts.
- Nature-based solutions or land cover indicators.

In [15]:
# Register each completed section output here.
# Future sections should append their metrics DataFrames to this list.
section_metric_tables = [
    population_metrics,
]

## 3. Combine And Export Context Metrics

This final section combines all completed context sections and writes one CSV for the selected country and administrative level.

In [16]:
identifier_columns = ["country_iso3", "country_name", "admin_level", "adm_id", "adm_name", "module", "hazard"]
context_metrics = admin_regions[identifier_columns].copy()

for metrics in section_metric_tables:
    if "adm_id" not in metrics.columns:
        raise ValueError("Each section metrics table must include an 'adm_id' column")
    context_metrics = context_metrics.merge(metrics, on="adm_id", how="left", validate="one_to_one")

# Round floating point outputs to keep the CSV readable. Counts remain model estimates, not integers.
numeric_columns = context_metrics.select_dtypes(include="number").columns
context_metrics[numeric_columns] = context_metrics[numeric_columns].round(3)

context_metrics.head()

,country_iso3,country_name,admin_level,adm_id,adm_name,module,hazard,pop_total,pop_female,pop_male,pop_under_5,pop_school_children_5_14,pop_working_age_15_64,pop_older_65_plus,pop_female_childbearing_15_49
0,KEN,Kenya,ADM1,32016919B72266624462344,Turkana,context,none,1074106.250,518061.122,556044.554,151482.689,305022.469,596109.132,21491.445,255517.524
1,KEN,Kenya,ADM1,32016919B63496705134089,Marsabit,context,none,615088.250,287648.498,327438.485,91460.078,180124.094,329090.162,14412.233,132223.533
2,KEN,Kenya,ADM1,32016919B2031803566233,Mandera,context,none,1032164.375,512839.463,519317.328,192278.977,345974.344,482180.639,11722.386,215100.789
3,KEN,Kenya,ADM1,32016919B89873713911655,Wajir,context,none,902046.312,420830.743,481212.464,141367.371,286673.984,463422.831,10578.886,196937.869
4,KEN,Kenya,ADM1,32016919B96045830258165,West Pokot,context,none,738739.625,371952.902,366780.400,134426.887,222575.203,364448.131,17283.093,163925.097


In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
context_metrics.to_csv(OUTPUT_PATH, index=False)

print(f"Exported {len(context_metrics):,} rows and {len(context_metrics.columns):,} columns to:")
print(OUTPUT_PATH)

Exported 47 rows and 15 columns to:
C:\Users\Mark.DESKTOP-UFHIN6T\Projects\national_tool_metrics\results\KEN\context\KEN_adm1_context_metrics.csv
